# Aggregate vectors → species-monthly

Consolidate `vectors_normalized.parquet` to H3 × month × species effort rows for vector-only training / analysis (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/02_interim/vectors_normalized.parquet` |
| Output | `data/03_processed/vectors_species_monthly.parquet` |
| Grain | `(h3_cell, year, month, scientific_name, vector_group)` |
| Measures | `individual_count` (sum), `observation_count`, `sampling_protocol_count` |

Rows missing any group key are dropped. Additive to `modelling_panel_species` — does not replace it.

Resolve repo paths and load the interim vector table.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import default_interim_dir, default_processed_dir, find_repo_root
from vector_guard.resolve import load_interim_vectors
from vector_guard.vectors_agg import (
    SPECIES_MONTHLY_COLS,
    SPECIES_MONTHLY_KEYS,
    aggregate_vectors_species_monthly,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = PROCESSED_DIR / "vectors_species_monthly.parquet"

vectors = load_interim_vectors(INTERIM_DIR)
(REPO_ROOT, OUT_PATH, len(vectors))

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/03_processed/vectors_species_monthly.parquet'),
 122789)

Aggregate to species-monthly grain, print key counts, write the parquet.

In [2]:
n_in = len(vectors)
key_ok = pd.Series(True, index=vectors.index)
for col in SPECIES_MONTHLY_KEYS:
    s = vectors[col]
    if pd.api.types.is_numeric_dtype(s):
        key_ok &= s.notna()
    else:
        as_str = s.astype("string")
        key_ok &= as_str.notna() & (as_str.str.strip() != "")
n_kept = int(key_ok.sum())
n_dropped = n_in - n_kept

agg = aggregate_vectors_species_monthly(vectors)
assert list(agg.columns) == SPECIES_MONTHLY_COLS

print(f"Input rows: {n_in}")
print(f"Rows with complete keys: {n_kept}")
print(f"Rows dropped (incomplete keys): {n_dropped}")
print(f"Output rows: {len(agg)}")
print(f"observation_count sum: {int(agg['observation_count'].sum())}")
print(f"individual_count sum: {float(agg['individual_count'].sum())}")
print(f"sampling_protocol_count max: {int(agg['sampling_protocol_count'].max())}")
assert int(agg["observation_count"].sum()) == n_kept

display(agg.head(10))
display(agg.describe(include="all"))

agg.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(agg)} rows)")

Input rows: 122789
Rows with complete keys: 121796
Rows dropped (incomplete keys): 993
Output rows: 32255
observation_count sum: 121796
individual_count sum: 8086067.0
sampling_protocol_count max: 4


,h3_cell,year,month,scientific_name,vector_group,individual_count,observation_count,sampling_protocol_count
0,8401265ffffffff,2015,7,Aedes diantaeus,mosquito,3,1,1
1,8401265ffffffff,2015,7,Aedes excrucians,mosquito,5,2,1
2,8401265ffffffff,2015,7,Aedes hexodontus,mosquito,9,2,2
3,8401265ffffffff,2015,7,Aedes pionips,mosquito,63,2,1
4,8401265ffffffff,2015,7,Aedes pullatus,mosquito,7,1,1
5,8401265ffffffff,2015,7,Aedes punctodes,mosquito,3,2,1
6,8401265ffffffff,2015,7,Aedes punctor,mosquito,9,2,1
7,8401269ffffffff,2016,12,Aedes communis,mosquito,6,1,1
8,8401269ffffffff,2016,12,Aedes diantaeus,mosquito,6,1,1
9,8401269ffffffff,2016,12,Aedes excrucians,mosquito,3,1,1


,h3_cell,year,month,scientific_name,vector_group,individual_count,observation_count,sampling_protocol_count
count,32255,32255.0,32255.0,32255,32255,32255.000000,32255.000000,32255.000000
unique,2200,<NA>,<NA>,246,4,NaN,NaN,NaN
top,8419519ffffffff,<NA>,<NA>,Culicoides,mosquito,NaN,NaN,NaN
freq,2305,<NA>,<NA>,4244,15067,NaN,NaN,NaN
mean,NaN,2012.144908,6.275709,NaN,NaN,250.691893,3.776035,1.029360
std,NaN,11.059354,3.085354,NaN,NaN,4569.745844,21.871072,0.180702
min,NaN,1957.0,1.0,NaN,NaN,0.000000,1.000000,1.000000
25%,NaN,2007.0,4.0,NaN,NaN,1.000000,1.000000,1.000000
50%,NaN,2016.0,6.0,NaN,NaN,2.000000,2.000000,1.000000
75%,NaN,2020.0,9.0,NaN,NaN,21.000000,3.000000,1.000000


Wrote /workspaces/vector-guard/data/03_processed/vectors_species_monthly.parquet (32255 rows)
